# Epic of Sun - Desafio 1
Benchmarking the models
***

This jupyter notebook benchmarks the models

It is divided in 2 parts:
* Loading libraries
* Benchmarking

***

## Loading libraries

In [1]:
import time
import numpy as np

import tensorflow as tf
from keras.models import load_model

from pathlib import Path
import json

2026-08-31 15:50:39.591702: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788202240.703589  127404 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788202241.178118  127404 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-31 15:50:44.691273: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Benchmarking

In [ ]:
# Select the type of model you want to analyze. You can choose between "unet", "fpn", and "attention_unet".
# The code will load the corresponding model and its training history based on your selection.
flag_model = 4

if flag_model == 1:
    model_type = "unet"
    model_type_name = "U-Net"
elif flag_model == 2:
    model_type = "fpn"
    model_type_name = "FPN"
elif flag_model == 3:
    model_type = "attention_unet"
    model_type_name = "Attention U-Net"
elif flag_model == 4:
    model_type = "deeplabv3"
    model_type_name = "DeepLabV3"

In [3]:
# Get the directory where THIS script is located
base_path = Path.cwd()
base_path = base_path.parent  # Move one level up to the project root

In [4]:
# Function to load the model from the specified path

def load_model_from_path(model_path):
    if model_path.exists():
        try:
            # Pass str(model_path) to ensure compatibility
            model = load_model(str(model_path), compile=False)
            file_name = model_path.name
            print(f"Successfully loaded {file_name} model.")
            return model
        except Exception as e:
            print(f"File {file_name} exists, but load_model failed with error:\n{e}")
    else:
        print(f"File {file_name} does not exist at the specified path. Check your working directory or base_path.")

In [5]:
# Function to benchmark the model

def benchmark_model(model, input_shape=(1, 128, 128, 7), n_warmup=10, n_runs=100):
    """
    Benchmarks a trained segmentation model for inference cost.
    input_shape: (batch, H, W, C) -- use batch=1 to simulate single-patch
    edge inference, which is the realistic deployment scenario.
    """
    dummy_input = tf.random.uniform(input_shape)

    # Warm-up: excludes one-time graph tracing/compilation overhead from the timing
    for _ in range(n_warmup):
        _ = model(dummy_input, training=False)

    # Timed runs
    latencies = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = model(dummy_input, training=False)
        latencies.append(time.perf_counter() - start)

    latencies = np.array(latencies) * 1000  # ms

    # Model size on disk (relevant for storage-constrained edge devices)
    n_params = model.count_params()
    param_bytes = sum(
        np.prod(w.shape) * np.dtype(str(w.dtype)).itemsize
        for w in model.trainable_weights
    )

    results = {
        "n_params": n_params,
        "param_size_mb": param_bytes / (1024 ** 2),
        "latency_mean_ms": latencies.mean(),
        "latency_p50_ms": np.percentile(latencies, 50),
        "latency_p95_ms": np.percentile(latencies, 95),
        "latency_std_ms": latencies.std(),
    }
    return results

In [6]:
# Function to print the results in a readable format
def print_benchmark_results(results, model_name="Model"):
    print(f"Benchmark results for {model_name}:")
    print(f"Number of parameters: {results['n_params']}")
    print(f"Parameter size (MB): {results['param_size_mb']:.2f}")
    print(f"Mean latency (ms): {results['latency_mean_ms']:.2f}")
    print(f"Median latency (ms): {results['latency_p50_ms']:.2f}")
    print(f"95th percentile latency (ms): {results['latency_p95_ms']:.2f}")
    print(f"Latency standard deviation (ms): {results['latency_std_ms']:.2f}")

    # Save benchmark data as a json file for future reference
    metrics_data = {
        "Model": model_name,
        "Number of parameters": results['n_params'],
        "Parameter size (MB)": results['param_size_mb'],
        "Mean latency (ms)": results['latency_mean_ms'],
        "Median latency (ms)": results['latency_p50_ms'],
        "95th percentile latency (ms)": results['latency_p95_ms'],
        "Latency standard deviation (ms)": results['latency_std_ms'],
    }

    metrics_path = base_path / "models" / f"{model_type}" / f"{model_type}_benchmarks_data.json"
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    with open(str(metrics_path), "w") as f:
        json.dump(metrics_data, f)
    print(f"Benchmark data saved to '{metrics_path}'")

In [7]:
# Load the models
model = load_model_from_path(base_path / "models" / f"{model_type}" / f"{model_type}_sugarcane.keras")

I0000 00:00:1788202283.759609  127404 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9367 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


Successfully loaded deeplabv3_sugarcane.keras model.


In [8]:
# Usage per architecture:
results = benchmark_model(model)

I0000 00:00:1788202385.922207  127404 cuda_dnn.cc:529] Loaded cuDNN version 90300


In [9]:
# Print the results
print_benchmark_results(results, model_name=model_type_name)

Benchmark results for DeepLabv3:
Number of parameters: 25449409
Parameter size (MB): 97.01
Mean latency (ms): 697.55
Median latency (ms): 128.19
95th percentile latency (ms): 5700.55
Latency standard deviation (ms): 1707.79
Benchmark data saved to '/home/fdesmello/Git/spacesugar2/models/deeplabv3/deeplabv3_benchmarks_data.json'
